# Prompt 14: Data Filtering and Row Manipulation
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (20%)

---

## filter() and where() — Identical Aliases

`filter()` and `where()` are **exact aliases** — they produce identical physical plans.
Use whichever reads more naturally in context.

```python
# Three equivalent forms of the same filter:
df.filter(col('salary') > 90000)
df.where(col('salary') > 90000)          # alias — identical
df.filter('salary > 90000')              # SQL string shortcut
df.where('salary > 90000')               # SQL string with where
```

## Boolean Operators for Compound Conditions

| Python operator | Spark operator | Example |
|----------------|----------------|--------|
| `&` | AND | `(col('a') > 1) & (col('b') < 5)` |
| `\|` | OR  | `(col('dept') == 'Eng') \| (col('dept') == 'HR')` |
| `~` | NOT | `~col('active')` |

**Critical rule:** Always wrap each condition in **parentheses** when combining with `&` / `|`.
Python operator precedence will produce `AnalysisException` without parens:

```python
# WRONG — operator precedence causes AnalysisException
df.filter(col('salary') > 90000 & col('active') == True)

# CORRECT — each condition in its own parentheses
df.filter((col('salary') > 90000) & (col('active') == True))
```

## Comparison Operators

| Python | Meaning |
|--------|---------|
| `==` | Equal |
| `!=` | Not equal |
| `>`, `>=` | Greater than / greater than or equal |
| `<`, `<=` | Less than / less than or equal |
| `.isin([v1, v2])` | IN list |
| `.between(low, high)` | Inclusive range |
| `.isNull()` / `.isNotNull()` | NULL check |
| `.startswith(s)` / `.endswith(s)` | String prefix/suffix |
| `.contains(s)` | String contains substring |
| `.like(pattern)` | SQL LIKE (`%` wildcard) |
| `.rlike(pattern)` | Regex LIKE |

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit

spark = SparkSession.builder \
    .appName('FilteringRowManipulation') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

data = [
    (1,  'Alice',   'Engineering', 95000.0,  8,  True),
    (2,  'Bob',     'Marketing',   72000.0,  3,  True),
    (3,  'Carol',   'Engineering', 110000.0, 12, False),
    (4,  'Dave',    'HR',           65000.0,  1,  True),
    (5,  'Eve',     'Engineering', 105000.0, 10, True),
    (6,  'Frank',   'Marketing',   None,      4,  True),
    (7,  'Grace',   'HR',           68000.0,  2,  False),
    (8,  'Henry',   'Engineering',  88000.0,  5,  True),
    (9,  'Iris',    'Marketing',    79000.0,  6,  True),
    (10, 'Jack',    'Finance',      91000.0,  7,  True),
]
schema = 'id INT, name STRING, department STRING, salary DOUBLE, years_exp INT, active BOOLEAN'
df = spark.createDataFrame(data, schema)
df.show()

In [ ]:
# Cell 2: filter() / where() — basic and compound conditions

print('=== filter() and where() are identical aliases ===')
df.filter(col('salary') > 90000).show()
df.where(col('salary') > 90000).show()   # same result

print('=== SQL string filter ===')
df.filter('salary > 90000').show()
df.where('salary > 90000 AND active = true').show()

print('=== AND with & operator — parens required! ===')
df.filter((col('salary') > 90000) & (col('active') == True)).show()

print('=== OR with | operator ===')
df.filter(
    (col('department') == 'Engineering') | (col('department') == 'Finance')
).show()

print('=== NOT with ~ operator ===')
df.filter(~col('active')).show()   # only inactive employees

print('=== Three-condition compound filter ===')
df.filter(
    (col('salary') > 80000) &
    (col('years_exp') >= 5) &
    (col('active') == True)
).show()

print('=== Demonstrating wrong parens (catches the error) ===')
try:
    # This is wrong — & binds tighter than > — would give AnalysisException or wrong result
    df.filter(col('salary') > 90000 & col('active') == True).show()
except Exception as e:
    print(f'Error (as expected): {type(e).__name__}')

In [ ]:
# Cell 3: isin(), between(), isNull(), isNotNull(), string filters

print('=== isin() — membership test ===')
df.filter(col('department').isin('Engineering', 'Finance')).show()

# isin() with a list variable
target_depts = ['Marketing', 'HR']
df.filter(col('department').isin(target_depts)).show()

# NOT isin
print('=== NOT isin — ~col.isin() ===')
df.filter(~col('department').isin('Engineering', 'Finance')).show()

print('=== between() — inclusive range ===')
df.filter(col('salary').between(70000, 95000)).show()
df.filter(col('years_exp').between(3, 7)).show()

print('=== isNull() and isNotNull() ===')
df.filter(col('salary').isNull()).show()     # Frank has null salary
df.filter(col('salary').isNotNull()).show()

print('=== String filters: startswith, endswith, contains, like, rlike ===')
df.filter(col('name').startswith('A')).show()
df.filter(col('name').endswith('e')).show()
df.filter(col('name').contains('a')).show()

print('=== like() — SQL LIKE with % and _ wildcards ===')
df.filter(col('name').like('A%')).show()   # starts with A
df.filter(col('name').like('%e')).show()   # ends with e
df.filter(col('name').like('_r%')).show()  # second char is 'r'

print('=== rlike() — regex filter ===')
df.filter(col('name').rlike('^[AEI]')).show()   # starts with A, E, or I
df.filter(col('name').rlike('.*e$')).show()      # ends with 'e'

## distinct() and dropDuplicates()

Both remove duplicate rows, but they differ in scope:

| Method | Scope | Syntax |
|--------|-------|--------|
| `distinct()` | All columns — row must be **100% identical** to be a duplicate | `df.distinct()` |
| `dropDuplicates()` | All columns — same as `distinct()` | `df.dropDuplicates()` |
| `dropDuplicates(['col1', 'col2'])` | Specified columns only — keeps first occurrence | `df.dropDuplicates(['dept', 'active'])` |
| `drop_duplicates()` | Alias for `dropDuplicates()` | `df.drop_duplicates(['col'])` |

**Key difference:**
- `distinct()` considers ALL columns — equivalent to `dropDuplicates()`
- `dropDuplicates(['col1'])` deduplicates based on **subset of columns** — keeps the first row seen for each unique combination of the specified columns (non-deterministic which row is kept without an explicit sort)

```python
# Keep only unique departments
df.select('department').distinct()

# Deduplicate by department+active, keep first occurrence
df.dropDuplicates(['department', 'active'])

# Deterministic dedup — sort first, then deduplicate
df.orderBy('salary', ascending=False).dropDuplicates(['department'])
# → keeps the highest-paid employee per department
```

**Exam tip:** `distinct()` triggers a **shuffle** (full sort/hash to find duplicates across partitions).

In [ ]:
# Cell 4: distinct() and dropDuplicates()

# Create a DataFrame with intentional duplicates
dup_data = [
    (1, 'Alice', 'Engineering', 95000.0),
    (1, 'Alice', 'Engineering', 95000.0),   # exact duplicate
    (2, 'Bob',   'Marketing',   72000.0),
    (3, 'Carol', 'Engineering', 110000.0),
    (4, 'Dave',  'Engineering', 88000.0),   # same dept as Carol but different row
    (5, 'Eve',   'Marketing',   79000.0),   # same dept as Bob
]
df_dup = spark.createDataFrame(dup_data, ['id', 'name', 'department', 'salary'])
print(f'Original row count: {df_dup.count()}')

print('=== distinct() — removes fully identical rows ===')
df_d = df_dup.distinct()
print(f'After distinct(): {df_d.count()} rows')
df_d.show()

print('=== dropDuplicates() — same as distinct() with no args ===')
df_dd = df_dup.dropDuplicates()
print(f'After dropDuplicates(): {df_dd.count()} rows')

print('=== dropDuplicates([subset]) — deduplicate by specific columns ===')
df_by_dept = df_dup.dropDuplicates(['department'])
print(f'After dropDuplicates(["department"]): {df_by_dept.count()} rows (one per dept)')
df_by_dept.show()

print('=== Deterministic dedup: sort first, then dropDuplicates ===')
df_top_earner = df_dup.orderBy('salary', ascending=False) \
                      .dropDuplicates(['department'])
print('Highest-paid per department:')
df_top_earner.show()

print('=== distinct() on single column — unique values ===')
df.select('department').distinct().show()
print(f'Unique departments: {df.select("department").distinct().count()}')

## Sorting — sort() and orderBy()

`sort()` and `orderBy()` are **identical aliases**.

```python
# Ascending (default)
df.sort('salary')
df.orderBy('salary')
df.orderBy(col('salary'))
df.orderBy(col('salary').asc())

# Descending
df.orderBy(col('salary').desc())
df.orderBy(col('salary'), ascending=False)  # for single column

# Multiple columns
df.orderBy('department', col('salary').desc())
df.orderBy(['department', 'salary'], ascending=[True, False])

# NULL ordering — control where NULLs appear
df.orderBy(col('salary').desc_nulls_last())    # NULLs at bottom
df.orderBy(col('salary').asc_nulls_first())    # NULLs at top
df.orderBy(col('salary').asc_nulls_last())     # NULLs at bottom
```

**Default NULL behaviour:**
- `asc()` — NULLs appear **first** (smallest)
- `desc()` — NULLs appear **last** (largest)

**Exam tip:** `sort()` / `orderBy()` on a distributed DataFrame triggers a **sort-based shuffle** and is expensive on large datasets. Avoid sorting entire DataFrames unless required for the final output.

## limit() — Take First N Rows

```python
df.limit(10)          # returns new DataFrame with at most 10 rows
df.sort('salary').limit(5)   # sort then limit — top 5 by salary
```

**limit() vs take() vs head():**

| Method | Returns | Lazy? |
|--------|---------|-------|
| `df.limit(n)` | New DataFrame (transformation) | Yes — lazy |
| `df.take(n)` | Python list of Row objects | No — action |
| `df.head(n)` | Python list of Row objects | No — action |
| `df.first()` | Single Row object | No — action |

In [ ]:
# Cell 5: sort() / orderBy() — ascending, descending, multi-column, null ordering

print('=== sort() and orderBy() are identical ===')
df.sort('salary').show()
df.orderBy('salary').show()   # identical

print('=== Descending ===')
df.orderBy(col('salary').desc()).show()

print('=== Multi-column sort ===')
df.orderBy('department', col('salary').desc()).show()

print('=== Multi-column with ascending list ===')
df.orderBy(['department', 'salary'], ascending=[True, False]).show()

print('=== NULL ordering — default asc puts NULLs first ===')
df.orderBy(col('salary').asc()).show()             # Frank (null) appears first

print('=== NULL ordering — asc_nulls_last ===')
df.orderBy(col('salary').asc_nulls_last()).show()  # Frank (null) appears last

print('=== NULL ordering — desc_nulls_last ===')
df.orderBy(col('salary').desc_nulls_last()).show() # highest first, null last

print('=== limit() — lazy transformation ===')
df_top3 = df.orderBy(col('salary').desc()).limit(3)
df_top3.show()
print(f'Row count after limit(3): {df_top3.count()}')

print('=== take() vs head() — actions returning Python lists ===')
rows = df.orderBy(col('salary').desc()).take(3)
print(f'take(3) returns: {type(rows)}, len={len(rows)}')
for r in rows:
    print(f'  {r.name}: {r.salary}')

first_row = df.first()
print(f'first() returns: {type(first_row)}, id={first_row.id}, name={first_row.name}')

## Filtering NULL Values

Three equivalent ways to filter nulls:

```python
# Keep rows where salary is NOT null
df.filter(col('salary').isNotNull())
df.filter(col('salary') != None)       # WRONG — never use Python None in filters!
df.filter('salary IS NOT NULL')        # SQL string
df.na.drop(subset=['salary'])          # dropna on subset

# Keep rows where salary IS null
df.filter(col('salary').isNull())
df.filter('salary IS NULL')
```

**Why `col('salary') != None` is wrong:**
- In SQL semantics, `NULL != NULL` evaluates to `NULL` (not `True`)
- Python `None` is not the same as Spark null in column comparisons
- Always use `.isNull()` / `.isNotNull()` methods

## Useful Row Count Patterns

```python
# Total rows
df.count()

# Rows where column is null
df.filter(col('salary').isNull()).count()

# Null count per column (aggregate)
from pyspark.sql.functions import count, when, isnan
df.select([F.count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Distinct count
df.select('department').distinct().count()
# or
df.agg(F.countDistinct('department')).show()
```

In [ ]:
# Cell 6: NULL filtering and row count patterns

print('=== isNull() / isNotNull() ===')
df.filter(col('salary').isNull()).show()    # Frank only
df.filter(col('salary').isNotNull()).show() # everyone else

print('=== SQL string null filter ===')
df.filter('salary IS NULL').show()
df.filter('salary IS NOT NULL').show()

print('=== Count rows with null salary ===')
null_count = df.filter(col('salary').isNull()).count()
print(f'Rows with null salary: {null_count}')

print('=== Null count for every column ===')
null_counts = df.select([
    F.count(F.when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
])
null_counts.show()

print('=== Combined null + value filter ===')
# Keep rows where salary IS NOT NULL AND > 80000
df.filter(col('salary').isNotNull() & (col('salary') > 80000)).show()

print('=== Distinct count ===')
print(f'Distinct departments: {df.select("department").distinct().count()}')
df.agg(F.countDistinct('department').alias('dept_count')).show()

print('=== Row counts summary ===')
print(f'Total rows:          {df.count()}')
print(f'Active employees:    {df.filter(col("active") == True).count()}')
print(f'Inactive employees:  {df.filter(~col("active")).count()}')
print(f'Null salaries:       {df.filter(col("salary").isNull()).count()}')
print(f'Unique departments:  {df.select("department").distinct().count()}')

In [ ]:
# Cell 7: Chaining filter + select + sort + limit — realistic query patterns

print('=== Pattern 1: Top 3 highest-paid active engineers ===')
df.filter((col('department') == 'Engineering') & (col('active') == True)) \
  .orderBy(col('salary').desc()) \
  .select('name', 'salary', 'years_exp') \
  .limit(3) \
  .show()

print('=== Pattern 2: Employees with 5-10 years exp in Eng or Finance, sorted by salary ===')
df.filter(
    col('department').isin('Engineering', 'Finance') &
    col('years_exp').between(5, 10)
) \
  .orderBy(col('salary').desc_nulls_last()) \
  .show()

print('=== Pattern 3: Non-null, non-zero salaries only ===')
df.filter(col('salary').isNotNull() & (col('salary') > 0)) \
  .orderBy('salary') \
  .select('name', 'department', 'salary') \
  .show()

print('=== Pattern 4: Employees whose name starts with vowel and dept contains "ing" ===')
df.filter(
    col('name').rlike('^[AEIOUaeiou]') &
    col('department').like('%ing')
) \
  .show()

print('=== Pattern 5: SQL-style filter via where() ===')
df.createOrReplaceTempView('employees')
spark.sql("""
    SELECT name, department, salary
    FROM employees
    WHERE salary > 80000
      AND active = true
    ORDER BY salary DESC
    LIMIT 5
""").show()

spark.stop()
print('Done.')

## Quick Reference Summary

### filter / where
```python
df.filter(col('x') > 5)          # Column expression
df.where(col('x') > 5)           # Identical alias
df.filter('x > 5')               # SQL string shortcut
# Compound: ALWAYS wrap each condition in parens
df.filter((col('a') > 1) & (col('b') < 5))
df.filter((col('a') > 1) | (col('b') < 5))
df.filter(~col('active'))         # NOT
```

### Column predicates
```python
col('dept').isin('A', 'B')        # IN list
col('dept').isin(my_list)         # IN from Python list
~col('dept').isin('A', 'B')       # NOT IN
col('val').between(low, high)      # inclusive range
col('x').isNull()                  # IS NULL
col('x').isNotNull()               # IS NOT NULL
col('s').startswith('A')           # string prefix
col('s').endswith('e')             # string suffix
col('s').contains('ar')            # substring
col('s').like('A%')                # SQL LIKE
col('s').rlike('^[A-Z]')           # regex
```

### Deduplication
```python
df.distinct()                      # all columns
df.dropDuplicates()                # same as distinct()
df.dropDuplicates(['col1'])        # subset of columns
df.drop_duplicates(['col1'])       # alias
# Deterministic: sort before dropDuplicates
df.orderBy('salary', ascending=False).dropDuplicates(['dept'])
```

### Sorting
```python
df.sort('col')                     # asc default
df.orderBy('col')                  # identical alias
df.orderBy(col('x').desc())
df.orderBy(col('x').asc_nulls_last())
df.orderBy(col('x').desc_nulls_last())
df.orderBy(['a', 'b'], ascending=[True, False])
```

### limit / take / head
```python
df.limit(n)     # transformation — returns DataFrame
df.take(n)      # action — returns list of Row
df.head(n)      # action — returns list of Row
df.first()      # action — returns single Row
```

## Real-World Scenarios

<details>
<summary>Scenario 1: Filtering a large event log for anomalies with compound conditions</summary>

**Situation:**
A security analyst queries a 500-million-row event log to find suspicious login events:
failed logins from non-whitelisted countries, outside business hours, not on VPN.

**Code:**
```python
whitelisted = ['US', 'UK', 'CA', 'AU']

suspicious = events.filter(
    (col('event_type') == 'LOGIN_FAILED') &
    (~col('country_code').isin(whitelisted)) &
    (col('hour_of_day').between(0, 6) | col('hour_of_day').between(22, 23)) &
    (col('vpn') == False)
) \
    .orderBy(col('timestamp').desc()) \
    .select('user_id', 'ip_address', 'country_code', 'timestamp', 'hour_of_day')

print(f'Suspicious logins: {suspicious.count()}')
suspicious.show(20)
```

**Exam Sub-topic:** Compound `&` / `|` / `~` conditions; `.isin()` with list; `.between()`; always wrap in parens
</details>

<details>
<summary>Scenario 2: Deduplicating CDC records — keep the latest version per key</summary>

**Situation:**
A Change Data Capture stream produces multiple updates per `customer_id`.
The pipeline needs the latest record per customer (highest `updated_at`).

**Code:**
```python
# Sort descending by updated_at, then deduplicate on customer_id
# dropDuplicates keeps the FIRST occurrence per key after the sort
latest_records = cdc_df \
    .orderBy(col('updated_at').desc_nulls_last()) \
    .dropDuplicates(['customer_id'])

print(f'Original CDC records:  {cdc_df.count()}')
print(f'Unique customers:      {latest_records.count()}')
```

**Exam Sub-topic:** `dropDuplicates(['col'])` with subset; sort before dedup for determinism; `desc_nulls_last()`
</details>

<details>
<summary>Scenario 3: Data quality gate — reject rows with nulls in required columns</summary>

**Situation:**
Before loading to the data warehouse, a pipeline splits incoming rows into
`valid` (all required columns non-null) and `quarantine` (any required column null) sets.

**Code:**
```python
required_cols = ['order_id', 'customer_id', 'amount', 'order_date']

# Build NOT-NULL condition for all required columns
not_null_condition = None
for c in required_cols:
    cond = col(c).isNotNull()
    not_null_condition = cond if not_null_condition is None else not_null_condition & cond

valid      = df.filter(not_null_condition)
quarantine = df.filter(~not_null_condition)

print(f'Valid rows:      {valid.count()}')
print(f'Quarantine rows: {quarantine.count()}')

valid.write.mode('append').delta('/warehouse/orders')
quarantine.write.mode('append').delta('/quarantine/orders')
```

**Exam Sub-topic:** `.isNotNull()`; dynamic compound filter built in loop; `~` NOT operator
</details>

<details>
<summary>Scenario 4: Pagination — implementing LIMIT/OFFSET for an API response</summary>

**Situation:**
A REST API endpoint returns paginated product search results.
The underlying Spark query must efficiently support page-based access.

**Code:**
```python
def get_page(df, page_number, page_size=20):
    """
    Returns a Python list of Row objects for the requested page.
    WARNING: OFFSET-based pagination is inefficient on large datasets
    (Spark must scan all preceding rows). Prefer cursor-based pagination.
    """
    offset = page_number * page_size
    return df.orderBy('product_id') \
             .limit(offset + page_size) \
             .tail(page_size)   # tail() is an action that returns last n rows

# Better: cursor-based pagination (filter on last seen id)
def get_next_page(df, last_seen_id, page_size=20):
    return df.filter(col('product_id') > last_seen_id) \
             .orderBy('product_id') \
             .limit(page_size)
```

**Exam Sub-topic:** `limit()` as transformation; `take()` / `head()` as actions; `first()`
</details>

<details>
<summary>Scenario 5: Filtering with regex to validate data format in a pipeline</summary>

**Situation:**
An inbound feed contains email addresses. The pipeline must separate well-formed
email addresses from malformed ones before processing.

**Code:**
```python
EMAIL_PATTERN = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

valid_emails   = df.filter(col('email').rlike(EMAIL_PATTERN))
invalid_emails = df.filter(~col('email').rlike(EMAIL_PATTERN) | col('email').isNull())

print(f'Valid:   {valid_emails.count()}')
print(f'Invalid: {invalid_emails.count()}')

# Further narrow: only @company.com emails
corporate = valid_emails.filter(col('email').endswith('@company.com'))
external  = valid_emails.filter(~col('email').endswith('@company.com'))
```

**Exam Sub-topic:** `.rlike()` for regex; `.endswith()`; `~` NOT; `.isNull()` combined with `|`
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| `filter()` vs `where()` | **Exact aliases** — identical physical plan |
| SQL string filter | `df.filter('salary > 90000 AND active = true')` — valid shortcut |
| `&` AND, `\|` OR, `~` NOT | Must wrap each condition in **parens** — precedence issue |
| Missing parens | Causes `AnalysisException` or wrong result |
| `col('x').isin('a', 'b')` | IN list; accepts `*args` or a Python list |
| `~col('x').isin('a')` | NOT IN |
| `col('x').between(lo, hi)` | Inclusive on both ends |
| `.isNull()` / `.isNotNull()` | Correct null check — do NOT use `== None` |
| `.like('A%')` | SQL LIKE — `%` = any chars, `_` = single char |
| `.rlike(pattern)` | Regex filter |
| `.startswith()` / `.endswith()` / `.contains()` | String predicates |
| `distinct()` | Remove 100%-identical rows; triggers shuffle |
| `dropDuplicates()` | Alias for `distinct()` with no args |
| `dropDuplicates(['col'])` | Dedup by subset; keeps first occurrence |
| Deterministic dedup | Sort **before** `dropDuplicates()` |
| `sort()` vs `orderBy()` | **Exact aliases** |
| `col('x').asc()` / `.desc()` | Sort direction modifiers |
| `asc_nulls_last()` / `desc_nulls_last()` | Control where NULLs appear |
| Default NULL in `asc()` | NULLs appear **first** |
| Default NULL in `desc()` | NULLs appear **last** |
| `limit(n)` | Transformation — returns DataFrame |
| `take(n)` / `head(n)` | Actions — return Python `list[Row]` |
| `first()` | Action — returns single `Row` |

---

## Practice Q&A

<details>
<summary>Q1: What is the difference between filter() and where()?</summary>

**A:** They are **exact aliases** — there is no difference whatsoever. Both accept a Column expression or SQL string. The physical plan produced is identical. `where()` is provided as a convenience for developers who prefer SQL-like naming. On the exam, any correct filter expression works with either keyword.
</details>

<details>
<summary>Q2: Why must conditions be wrapped in parentheses when using & and |?</summary>

**A:** Python's operator precedence gives `&` and `|` **higher priority** than comparison operators (`>`, `==`, `<`, etc.). Without parens, `col('salary') > 90000 & col('active') == True` is parsed as `col('salary') > (90000 & col('active')) == True`, which is nonsensical and raises `AnalysisException`. The fix is always: `(col('salary') > 90000) & (col('active') == True)`.
</details>

<details>
<summary>Q3: What is the difference between distinct() and dropDuplicates(['col'])?</summary>

**A:**
- `distinct()` — removes rows that are **100% identical across ALL columns**. Two rows must match on every column to be considered duplicates.
- `dropDuplicates(['col'])` — removes rows based on **specified columns only**, keeping the first occurrence. Other columns can differ — only the subset matters for deduplication.

Example: rows `(1, 'Alice', 'Eng')` and `(2, 'Alice', 'Eng')` are NOT removed by `distinct()` (different `id`), but ARE removed by `dropDuplicates(['name', 'department'])`.
</details>

<details>
<summary>Q4: What is the default NULL ordering for asc() and desc()?</summary>

**A:**
- `asc()` — NULLs appear **first** (treated as smallest value)
- `desc()` — NULLs appear **last** (treated as smallest, so they go to the bottom when descending)

To override: use `asc_nulls_last()`, `asc_nulls_first()`, `desc_nulls_first()`, `desc_nulls_last()`.
</details>

<details>
<summary>Q5: What is the difference between limit() and take()?</summary>

**A:**
- `df.limit(n)` — a **transformation** that returns a new **DataFrame** with at most n rows. It is lazy — no data is processed until an action is called.
- `df.take(n)` — an **action** that triggers execution and returns a **Python list of Row objects**. The result is collected to the driver.
- `df.head(n)` — identical to `take(n)`.
- `df.first()` — action returning a single `Row` (equivalent to `take(1)[0]`).

Use `limit()` when you need to pass a smaller DataFrame to a downstream transformation. Use `take()`/`head()` when you need to inspect the data in Python.
</details>